<a href="https://colab.research.google.com/github/nakamura196/000_tools/blob/main/IIIF%E3%83%9E%E3%83%8B%E3%83%95%E3%82%A7%E3%82%B9%E3%83%88%E3%83%95%E3%82%A1%E3%82%A4%E3%83%AB%E3%81%8B%E3%82%89TEI_XML%E3%83%95%E3%82%A1%E3%82%A4%E3%83%AB%E3%82%92%E4%BD%9C%E6%88%90%E3%81%99%E3%82%8B%E3%83%97%E3%83%AD%E3%82%B0%E3%83%A9%E3%83%A0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IIIFマニフェストファイルからTEI/XMLファイルを作成するプログラム

IIIFマニフェストファイルのURLを指定して、NDL古典籍OCR-LiteによるOCR結果を含むTEI/XMLファイルを作成します。

In [ ]:
# @title 準備

%cd /content
!rm -rf ndlkotensekiocr_tools
!git clone https://github.com/nakamura196/ndlkotensekiocr_tools

%cd ndlkotensekiocr_tools

!git clone https://github.com/ndl-lab/ndlkotenocr-lite

!pip install -qr ndlkotenocr-lite/requirements.txt

# ndlkotenocr-lite のバグを修正
ocr_file_path = "/content/ndlkotensekiocr_tools/ndlkotenocr-lite/src/ocr.py"

with open(ocr_file_path, "r", encoding="utf-8") as f:
    ocr_code = f.read()

patches_applied = []

# パッチ1: ZeroDivisionError を修正
if "if tatelinecnt/alllinecnt>0.5:" in ocr_code:
    ocr_code = ocr_code.replace(
        "if tatelinecnt/alllinecnt>0.5:",
        "if alllinecnt > 0 and tatelinecnt/alllinecnt>0.5:"
    )
    patches_applied.append("ZeroDivisionError修正")

# パッチ2: 検出モデルの重複初期化を修正 + 進捗表示
cache_code = '''
# モデルキャッシュ（パッチ追加）
import sys as _sys
_detector_cache = None
_image_counter = 0
_total_images = 0

def get_detector_cached(args):
    global _detector_cache
    if _detector_cache is None:
        print("[INFO] Intialize Model (cached)")
        _detector_cache = get_detector(args)
    return _detector_cache

def set_total_images(total):
    global _total_images, _image_counter
    _total_images = total
    _image_counter = 0

def print_progress():
    global _image_counter, _total_images
    _image_counter += 1
    # シンプルな進捗表示（10件ごとに改行）
    _sys.stdout.write(f" {_image_counter}/{_total_images}")
    if _image_counter % 10 == 0 or _image_counter == _total_images:
        _sys.stdout.write("\\n")
    _sys.stdout.flush()
'''

if "_detector_cache = None" not in ocr_code and "def get_detector(" in ocr_code:
    if "def inference_on_detector(" in ocr_code:
        ocr_code = ocr_code.replace(
            "def inference_on_detector(",
            cache_code + "\ndef inference_on_detector("
        )
        patches_applied.append("検出モデルキャッシュ追加")

        if "detector=get_detector(args)" in ocr_code:
            ocr_code = ocr_code.replace(
                "detector=get_detector(args)",
                "detector=get_detector_cached(args)"
            )
            patches_applied.append("検出モデル再利用")

# パッチ3: 重複ログ抑制
if 'print("[INFO] Intialize Model")' in ocr_code and "_detector_cache" in ocr_code:
    ocr_code = ocr_code.replace(
        'print("[INFO] Intialize Model")',
        '# print("[INFO] Intialize Model")  # キャッシュ版で表示'
    )
    patches_applied.append("重複ログ抑制")

# パッチ4: 進捗表示をコンパクト形式に変更
if 'print("[INFO] Inference Image")' in ocr_code and "print_progress" in ocr_code:
    ocr_code = ocr_code.replace(
        'print("[INFO] Inference Image")',
        'print_progress()'
    )
    patches_applied.append("進捗表示改善")

# パッチ5: ループ前に総画像数を設定と開始メッセージ
if "for inputpath in inputpathlist:" in ocr_code and "set_total_images" in ocr_code:
    ocr_code = ocr_code.replace(
        "for inputpath in inputpathlist:",
        'set_total_images(len(inputpathlist))\n    print(f"[INFO] Processing {_total_images} images:")\n    for inputpath in inputpathlist:'
    )
    patches_applied.append("総画像数カウント追加")

with open(ocr_file_path, "w", encoding="utf-8") as f:
    f.write(ocr_code)

if patches_applied:
    print(f"✓ ndlkotenocr-lite に以下のパッチを適用しました:")
    for patch in patches_applied:
        print(f"  - {patch}")
else:
    print("⚠️ パッチ対象のコードが見つかりませんでした（既に修正済みの可能性）")

from ndlkotensekiocr_tools.core import CoreClient
import subprocess
import sys
import os
from glob import glob
import xml.etree.ElementTree as ET

# エラー情報を保存するリスト
processing_errors = []

# OCR進捗表示のためにrun_ocrメソッドをオーバーライド
def run_ocr_with_progress(self, input_dir, output_dir, src_path):
    """OCR処理の実行（リアルタイム進捗表示付き）"""
    current_path = os.getcwd()
    input_abs_path = os.path.abspath(input_dir)
    output_abs_path = os.path.abspath(output_dir)

    try:
        os.chdir(src_path)

        env = os.environ.copy()
        env['PYTHONUNBUFFERED'] = '1'

        process = subprocess.Popen(
            ["python", "-u", "ocr.py", "--sourcedir", input_abs_path, "--output", output_abs_path],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env
        )

        for line in iter(process.stdout.readline, ''):
            print(line, end='', flush=True)
            sys.stdout.flush()

        process.wait()

    finally:
        os.chdir(current_path)


def validate_ocr_file(file_path):
    """OCRファイルが有効かどうかを検証"""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()
            if not content.strip():
                return False, "ファイルが空です"

            page_xml = ET.fromstring(content)
            page_info = page_xml.find(".//PAGE")

            if page_info is None:
                return False, "PAGE要素がありません"

            ocr_page_width = int(page_info.get("WIDTH", "0"))
            if ocr_page_width == 0:
                return False, "ページ幅が0です"

            lines = page_xml.findall(".//LINE")
            if len(lines) == 0:
                return False, "テキストなし（表紙・白紙・挿絵等）"

            return True, None
    except ET.ParseError as e:
        return False, f"XMLパースエラー: {e}"
    except Exception as e:
        return False, f"検証エラー: {e}"


def create_tei_xml_safe(self, manifest_url, input_dir, output_path):
    """TEI/XML作成（エラーハンドリング強化版・空ページ対応）"""
    from tqdm import tqdm
    from datetime import datetime
    from xml.dom import minidom

    if not self.manifest_data:
        self._fetch_manifest(manifest_url)

    files = sorted(glob(f"{input_dir}/*.xml"))

    if self.end_index:
        files = files[:self.end_index]

    # 検証: OCRファイルが存在するか
    if len(files) == 0:
        raise ValueError(f"OCR結果ファイルが見つかりません: {input_dir}")

    # 検証: キャンバス数とOCRファイル数の比較
    num_canvases = len(self.canvases)
    num_ocr_files = len(files)

    if num_ocr_files > num_canvases:
        print(f"⚠️ 警告: OCRファイル数({num_ocr_files})がキャンバス数({num_canvases})より多いです")
        print(f"   最初の{num_canvases}ファイルのみ処理します")
        files = files[:num_canvases]

    # TEI作成
    tei = self._create_tei_base(manifest_url)
    body = tei.find(".//body")
    facsimile = tei.find(".//facsimile")

    processed_with_text = 0
    processed_empty = 0
    error_pages = []

    # 全てのファイルを処理（空ページも含む）
    for i, file_path in enumerate(tqdm(files, desc="Creating TEI/XML")):
        try:
            # インデックスがキャンバス範囲内かチェック
            if i >= len(self.canvases):
                print(f"⚠️ スキップ: インデックス {i} がキャンバス範囲外です")
                continue

            canvas = self.canvases[i]
            canvas_width = canvas["images"][0]["resource"]["width"]

            if canvas_width == 0:
                print(f"⚠️ スキップ: ページ {i+1} のキャンバス幅が0です")
                continue

            # surface要素は常に作成（空ページでも）
            surface = self._create_surface(canvas, i, facsimile)

            # ab要素も常に作成（空ページでも）
            div = self._create_page_div(body, i, canvas)

            # OCRファイルの検証
            is_valid, error_msg = validate_ocr_file(file_path)

            if not is_valid:
                # テキストがないページ: surface と ab は作成済み、中身は空のまま
                print(f"📄 ページ{i+1}: {error_msg}（空のab/surface要素を作成）")
                processed_empty += 1
                continue

            # 有効なOCRファイルの処理
            with open(file_path, "r", encoding="utf-8") as f:
                page_xml = ET.fromstring(f.read())

            page_info = page_xml.find(".//PAGE")
            ocr_page_width = int(page_info.get("WIDTH", "0"))

            if ocr_page_width == 0:
                print(f"📄 ページ{i+1}: OCR幅が0（空のab/surface要素を作成）")
                processed_empty += 1
                continue

            # LINE要素を処理
            for line in page_xml.findall(".//LINE"):
                line_x = int(line.get("X", "0"))
                line_y = int(line.get("Y", "0"))
                line_width = int(line.get("WIDTH", "0"))
                line_height = int(line.get("HEIGHT", "0"))

                ratio = canvas_width / ocr_page_width
                fixed_x = int(line_x * ratio)
                fixed_y = int(line_y * ratio)
                fixed_width = int(line_width * ratio)
                fixed_height = int(line_height * ratio)

                line_index = str(int(line.get("ORDER", "0")) + 1)

                zone = self._create_zone(surface, line, fixed_x, fixed_y,
                                    fixed_width, fixed_height, line_index)

                self._add_text_elements(div, line, line_index, zone)

            processed_with_text += 1

        except Exception as e:
            print(f"⚠️ ページ {i+1} の処理中にエラー: {e}")
            error_pages.append(i+1)
            continue

    total_processed = processed_with_text + processed_empty

    if total_processed == 0:
        raise ValueError("有効なページを処理できませんでした")

    # XML出力
    self._write_tei_xml(tei, output_path)

    print(f"✓ TEI/XML作成完了: {output_path}")
    print(f"   - テキストあり: {processed_with_text}ページ")
    print(f"   - テキストなし（空要素）: {processed_empty}ページ")
    print(f"   - 合計: {total_processed}ページ")

    if error_pages:
        print(f"⚠️ エラー発生ページ: {error_pages}")


def main_safe(manifest_url, output_dir, src_path, verbose=False, resized_width=1200, interval=1, end_index=None):
    """安全なmain関数（エラーハンドリング強化版）"""
    client = CoreClient(end_index=end_index)
    paths = client._setup_directories(output_dir)

    item_id = output_dir.split('/')[-1]
    print(f"\n{'='*50}")
    print(f"処理開始: {item_id}")
    print(f"{'='*50}")

    try:
        # 1. 画像ダウンロード
        print("\n[1/3] 画像ダウンロード中...")
        client.download_images(
            manifest_url=manifest_url,
            output_dir=paths['img_dir'],
            verbose=verbose,
            resized_width=resized_width,
            interval=interval
        )

        img_files = glob(f"{paths['img_dir']}/*.jpg")
        if len(img_files) == 0:
            raise ValueError("画像がダウンロードされませんでした")
        print(f"   ✓ {len(img_files)}枚の画像をダウンロード")

        # 2. OCR実行
        print("\n[2/3] OCR処理中...")
        client.run_ocr(
            input_dir=paths['img_dir'],
            output_dir=paths['ocr_dir'],
            src_path=src_path
        )

        ocr_files = glob(f"{paths['ocr_dir']}/*.xml")
        if len(ocr_files) == 0:
            raise ValueError("OCR結果が生成されませんでした")
        print(f"   ✓ {len(ocr_files)}件のOCR結果")

        # 3. TEI/XML作成
        print("\n[3/3] TEI/XML作成中...")
        create_tei_xml_safe(
            client,
            manifest_url=manifest_url,
            input_dir=paths['ocr_dir'],
            output_path=paths['xml_path']
        )

        print(f"\n✅ {item_id} の処理が完了しました")
        return True, None

    except Exception as e:
        error_msg = str(e)
        print(f"\n❌ {item_id} の処理中にエラーが発生しました: {error_msg}")

        if os.path.exists(paths['xml_path']):
            try:
                file_size = os.path.getsize(paths['xml_path'])
                if file_size < 1000:
                    os.remove(paths['xml_path'])
                    print(f"   空のXMLファイルを削除しました: {paths['xml_path']}")
            except:
                pass

        processing_errors.append({
            'item_id': item_id,
            'manifest_url': manifest_url,
            'error': error_msg
        })
        return False, error_msg


# メソッドを置き換え
CoreClient.run_ocr = run_ocr_with_progress
CoreClient.main = main_safe

client = CoreClient()
print("✓ 準備完了（エラーハンドリング強化版）")

In [ ]:
# @title 実行
manifest_url = "https://iiif.dl.itc.u-tokyo.ac.jp/repo/iiif/0f11a3ed-18c2-7322-6340-19ed3f0d966e/manifest" # @param {"type":"string"}
output_dir = "/content/data/utokyo-ogai-01" # @param {"type":"string"}

# manifest_url = "https://kokusho.nijl.ac.jp/biblio/100421273/manifest"
# id = "nijl-100421273"
# output_dir = f"./tmp/data4/{id}"

CoreClient.main(manifest_url, output_dir, "/content/ndlkotensekiocr_tools/ndlkotenocr-lite/src")

### 生成されたTEI/XMLファイルの一括ダウンロード

`/content/data`ディレクトリとそのサブディレクトリにあるすべてのXMLファイルを検索し、単一のzipアーカイブに圧縮します。その後、ダウンロードリンクが提供されます。

In [ ]:
# @title 実行

import os
import zipfile
from google.colab import files

output_base_dir = "/content/data"
zip_filename = "tei_xml_files_merged_only.zip"

merged_xml_files_found = []

# output_base_dir内の各ディレクトリを走査
for item_id_dir in os.listdir(output_base_dir):
    item_path = os.path.join(output_base_dir, item_id_dir)

    # それがディレクトリであることを確認
    if os.path.isdir(item_path):
        # 期待されるマージ済みXMLファイルのパスを構築
        final_xml_filename = f"{item_id_dir}.xml"
        final_xml_path = os.path.join(item_path, final_xml_filename)

        # ファイルが存在するか確認
        if os.path.exists(final_xml_path):
            merged_xml_files_found.append(final_xml_path)

if not merged_xml_files_found:
    print(f"Warning: No merged XML files found in '{output_base_dir}' matching the pattern `/{item_id_dir}/{item_id_dir}.xml`.")
else:
    # zipファイルを作成
    with zipfile.ZipFile(zip_filename, 'w') as zipf:
        for file_path in merged_xml_files_found:
            # ファイルのベース名（例: tohakugenji_01.xml）をarcnameとして使用
            arcname = os.path.basename(file_path)
            zipf.write(file_path, arcname)
            print(f"Added {arcname} to zip.")

    print(f"All {len(merged_xml_files_found)} merged XML files have been zipped into '{zip_filename}'.")

    # zipファイルをダウンロード
    files.download(zip_filename)
